# KNN - Tenant Preference Prediction

Project problem: predict the preferred tenant type for a rental house listing.

In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import joblib

In [2]:
df = pd.read_csv('../House_Rent_Dataset_TN.csv')
df.head()

,Rent,Posted On,BHK,Size,Floor,Area Type,Area Locality,City,Furnishing Status,Tenant Preferred,...,Listing_Description,Amenities_Text,Landmarks_Text,Reviews_Text,Complaints_Text,Lease_Rules_Text,Inquiry_Text,Inspection_Notes_Text,Scam_Flag,Scam_Rationale
0,15000,2022-07-06,2,1100,1 out of 2,super area,medavakkam,chennai,semi-furnished,bachelors,...,2 bhk semi-furnished home of 1100 sq ft (super...,"rainwater harvesting, play area, solar lightin...",near college (3.3 km); near hospital (0.6 km);...,clean and well maintained building. | safe are...,not_applicable,no smoking inside the house. tenant must follo...,looking for a 2 bhk near medavakkam within inr...,flooring shows normal wear. bathrooms are clea...,no,not_applicable
1,6500,2022-05-21,2,1000,ground out of 1,super area,"urapakkam, vandalur r.f, gst road",chennai,semi-furnished,bachelors/family,...,2 bhk semi-furnished home of 1000 sq ft (super...,"visitor parking, cctv, power backup, club hous...",near market (3.5 km); near school (0.8 km); ne...,clean and well maintained building. | good wat...,listing photos did not match actual house.,security deposit is two months rent. no loud m...,"want to move in next month, budget around inr ...",flooring shows normal wear. good natural light...,yes,owner refused physical visit. | asked for paym...
2,90000,2022-05-20,3,2400,1 out of 3,carpet area,"r.a puram, mandaiveli",chennai,semi-furnished,bachelors/family,...,3 bhk semi-furnished home of 2400 sq ft (carpe...,"waste management, power backup, lift, fire saf...",near pharmacy (0.6 km); near church (0.6 km),good water supply and quiet neighborhood. | ma...,not_applicable,security deposit is two months rent. maintenan...,"want to move in next month, budget around inr ...",electrical fittings are working properly. floo...,no,not_applicable
3,200000,2022-07-10,3,3000,1 out of 1,super area,madras boat club road,chennai,furnished,family,...,3 bhk furnished home of 3000 sq ft (super area...,"security, play area, gym, maintenance staff, w...",near mosque (3.4 km); near atm (1.8 km),parking is convenient and secure. | spacious r...,not_applicable,maintenance charges are extra. pets allowed on...,"want to move in next month, budget around inr ...",walls need minor repainting. good natural ligh...,no,not_applicable
4,15000,2022-06-25,1,650,ground out of 2,carpet area,kambar colony,chennai,semi-furnished,bachelors/family,...,1 bhk semi-furnished home of 650 sq ft (carpet...,"maintenance staff, security, rainwater harvest...",near atm (3.5 km); near hospital (3.8 km); nea...,easy access to public transport. | maintenance...,not_applicable,pets allowed only with prior approval. no loud...,"need a semi-furnished home in chennai, prefer ...",bathrooms are clean and functional. walls need...,no,not_applicable


In [3]:
df.shape

(1500, 23)

In [4]:
df['Tenant Preferred'].value_counts()

Tenant Preferred
bachelors/family    1049
bachelors            244
family               207
Name: count, dtype: int64

## Select input and output columns

In [5]:
X = df[['Rent', 'BHK', 'Size', 'Bathroom', 'Area Type', 'City', 'Furnishing Status']]
y = df['Tenant Preferred']

X.head()

,Rent,BHK,Size,Bathroom,Area Type,City,Furnishing Status
0,15000,2,1100,2,super area,chennai,semi-furnished
1,6500,2,1000,2,super area,chennai,semi-furnished
2,90000,3,2400,3,carpet area,chennai,semi-furnished
3,200000,3,3000,4,super area,chennai,furnished
4,15000,1,650,1,carpet area,chennai,semi-furnished


## Convert text columns into numbers

In [6]:
X = pd.get_dummies(X, drop_first=True)
X.head()

,Rent,BHK,Size,Bathroom,Area Type_carpet area,Area Type_super area,City_coimbatore,Furnishing Status_semi-furnished,Furnishing Status_unfurnished
0,15000,2,1100,2,False,True,False,True,False
1,6500,2,1000,2,False,True,False,True,False
2,90000,3,2400,3,True,False,False,True,False
3,200000,3,3000,4,False,True,False,False,False
4,15000,1,650,1,True,False,False,True,False


## Split the data

In [7]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

## Train KNN model

In [8]:
knn = KNeighborsClassifier(n_neighbors=5)
knn.fit(X_train, y_train)

,n_neighbors,5
,weights,'uniform'
,algorithm,'auto'
,leaf_size,30
,p,2
,metric,'minkowski'
,metric_params,None
,n_jobs,None


## Check accuracy

In [13]:
y_pred = knn.predict(X_test)

print('Accuracy:', accuracy_score(y_test, y_pred))
print('\nClassification Report:')
print(classification_report(y_test, y_pred))

Accuracy: 0.68

Classification Report:
                  precision    recall  f1-score   support

       bachelors       0.30      0.28      0.29        47
bachelors/family       0.77      0.83      0.80       218
          family       0.43      0.26      0.32        35

        accuracy                           0.68       300
       macro avg       0.50      0.46      0.47       300
    weighted avg       0.66      0.68      0.67       300



In [10]:
confusion_matrix(y_test, y_pred)

array([[ 13,  32,   2],
       [ 26, 182,  10],
       [  4,  22,   9]])

## Sample prediction

In [11]:
print('Predicted:', knn.predict(X_test.head()))
print('Actual:')
print(y_test.head().values)

Predicted: ['bachelors/family' 'bachelors/family' 'bachelors/family'
 'bachelors/family' 'bachelors/family']
Actual:
<ArrowStringArray>
[          'family', 'bachelors/family', 'bachelors/family',
 'bachelors/family', 'bachelors/family']
Length: 5, dtype: str


## Save model

In [12]:
joblib.dump(knn, 'house_rent_knn_tenant_preference.joblib')

['house_rent_knn_tenant_preference.joblib']